# RAPID response-prediction investigation

This notebook evaluates continuous prediction of the signed RAPID timing response from geometry. It uses grouped leave-one-example-out validation and never uses the timing response as an input feature.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'RAPID').exists():
    REPO_ROOT = REPO_ROOT.parents[1]
OUTPUT_ROOT = Path('/Volumes/PhD/river_hierarchy/network_variants/output')
EXAMPLE_IDS = ['02', '03', '05', '11','13','15', '21', '56']  # edit when more real-network outputs are available
RUN_DIR = REPO_ROOT / 'RAPID' / 'classification' / 'results' / f"prediction-notebook-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"


In [ ]:
command = [sys.executable, str(REPO_ROOT / 'RAPID/classification/train_rapid_response_regressor.py'),
           '--output-root', str(OUTPUT_ROOT), '--results-dir', str(RUN_DIR),
           '--example-ids', *EXAMPLE_IDS]
subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
scores = pd.read_csv(RUN_DIR / 'grouped_cv_regression_scores.csv')
predictions = pd.read_csv(RUN_DIR / 'grouped_cv_regression_predictions.csv')
display(scores)


In [ ]:
recommended = scores.iloc[0]['model']
plot_df = predictions.loc[predictions.model.eq(recommended)]
fig, ax = plt.subplots(figsize=(6, 6))
for example_id, group in plot_df.groupby('example_id'):
    ax.scatter(group.observed_lag_shift_pct, group.predicted_lag_shift_pct, label=example_id, alpha=.8)
limits = [min(plot_df.observed_lag_shift_pct.min(), plot_df.predicted_lag_shift_pct.min()),
          max(plot_df.observed_lag_shift_pct.max(), plot_df.predicted_lag_shift_pct.max())]
ax.plot(limits, limits, 'k--', label='perfect prediction')
ax.set(xlabel='Observed RAPID lag shift [%]', ylabel='Predicted lag shift [%]', title=f'{recommended}: leave-one-example-out predictions')
ax.legend(title='Held-out example')
plt.show()
display(plot_df.groupby('example_id').agg(n=('state_id','size'), mae=('residual_pct_points', lambda x: x.abs().mean()), bias=('residual_pct_points','mean')))
